# Holter 증강: Minimum-overlap Cropping (v2 Swee 파이프라인용)

**목적**: 원본 Holter 하나에서 **서로 다른 시간대를 최대한 담은 15시간 이상 구간을 최대 2개** 잘라 새 WFDB 파일로 저장한다. 무작위로 자르지 않고, 두 증강 구간의 겹침이 가장 작게 되도록 **기록 맨 앞 15시간과 맨 뒤 15시간**을 쓴다.

```
24시간 원본:  0h ─────────────────────────── 24h
증강 -1:      0h ───────────────── 15h
증강 -2:              9h ─────────────────── 24h      겹침 6h (이론적 최소 = 15 + 15 − 24)
```

| 항목 | 내용 |
|---|---|
| 대상 | 최종 사용 환자 445명 (`final_holter_ecg_1_cohort.ipynb` ⑥ → `cohort_final.csv`) |
| 원본 | `/mnt/c/data/music/Holter_ECG` (수정하지 않음) |
| 증강 저장 | `/home/aiuser/data/music/Holter_ECG_aug` (`P0001-1`, `P0001-2`) |
| 저장 방식 | 원본의 16비트 정수 샘플을 그대로 잘라 같은 형식(format 16, gain, baseline, 유도 순서)으로 저장 → **값 손실 없음** |
| 기록 | `results/final_holter_ecg/` 아래 계획(dry-run), 메타데이터, 오류, 검증 결과 CSV |

## 학습할 때 쓰는 규칙 (데이터 누수 방지)
(비워 둠. 학습 방식이 정해지면 채움)

## 결정 기록 (사용자가 정한 것만)

| 날짜 | 결정 |
|---|---|
| 2026-10-26 | v2(Swee) 파이프라인에 Minimum-overlap Cropping 증강 적용. 최소 15시간, 원본당 최대 2개, 무작위 자르기 안 함 |
| 2026-10-26 | 대상은 Swee 코호트 730명, 저장 위치는 `/home/aiuser/data/music/Holter_ECG_aug`, 3유도 그대로 저장 |
| 2026-10-26 | 노트북으로 작성 (설정 → dry-run → 소수 시험 → 검증 → 전체) |
| 2026-10-27 | 증강 대상을 730명에서 최종 사용 환자 445명(`cohort_final.csv`)으로 바꿈 (팀 분할 파일에서 등록 전 기기 이식 63명 제외) |
| 2026-10-27 | 교육용 노트북은 용량이 부족해 원본·전처리·증강 파일을 저장할 수 없음 → 전처리와 증강은 집 컴퓨터에서 하고, 결과를 구글 드라이브에 올려서 사용 |

## 검토 중 (제안, 아직 결정 안 함)

| 항목 | 내용 |
|---|---|
| 증강 효과 비교 | 증강 없는 v2 결과(Swee ECG 단독 SCD AUROC 0.628)와 같은 환자 분할로 비교해야 효과를 말할 수 있음 |
| 15시간 근거 | 요청서 값. 문헌 근거는 아직 찾지 않음 |
| 중복률 상한 | 기본은 제한 없음(`MAX_OVERLAP_RATIO = None`). 중복률 높은 환자는 ⑨에 따로 표시 |

## 알아 둘 점
- 대상을 445명으로 바꾸기 전에 730명 기준으로 ⑥ 시험을 돌린 컴퓨터에는 445명 밖 환자의 증강 파일 `P0906-1`, `P0906-2`가 남아 있을 수 있다. ⑧의 '계획에 없는 증강 파일'에 나오면 지울지 직접 판단
- **기록 맨 앞 약 30초는 1 mV 보정 신호(사각파)**다 (⑥-2 그림, 확인한 5명 중 4명 30초, 1명 10초). 증강 `-1`은 0시간부터 시작하므로 이 신호가 그대로 들어 있다. v2 전처리의 "앞 30초 버림"이 이 부분을 지운다. 증강 `-2`는 기록 중간에서 시작하므로 보정 신호가 없고, 앞 30초 버림으로 실제 심전도 30초가 빠진다 (15시간 중 0.06%).
- 증강 `-1`은 원본의 앞 15시간과 같다. 그래서 증강본에 v2 전처리를 돌리면 **원본 특징 묶음의 부분 묶음**에 가깝다. 증강 `-2`는 시작 위치가 30초 단위가 아니어서 조각 경계가 밀리는 만큼만 값이 달라진다.
- 헤더는 원본과 정보가 같지만 표기가 조금 다르다: 단위 `mV`가 명시되고, 유도마다 초기값과 체크섬이 추가된다 (wfdb가 쓰는 표준 형식).

In [ ]:
# ① 설정: 경로, 증강 규칙, 실행 스위치
from pathlib import Path
import hashlib
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wfdb
from tqdm.auto import tqdm

plt.rcParams["font.family"] = "NanumGothic"       # 그림의 한글 (다른 전처리 노트북과 같은 글꼴)
plt.rcParams["axes.unicode_minus"] = False

# 경로
RAW_DIR = Path("/mnt/c/data/music/Holter_ECG")                     # 원본 (읽기만 함)
RECORDS_FILE = RAW_DIR.parent / "RECORDS"                          # PhysioNet 기록 목록 = 원본 목록의 기준
AUG_DIR = Path("/home/aiuser/data/music/Holter_ECG_aug")           # 증강 파일 저장 위치
TMP_DIR = AUG_DIR / ".tmp"                                         # 쓰는 중인 파일 (다 쓰면 AUG_DIR로 옮김)
COHORT_PATH = Path("../results/final_holter_ecg/cohort_final.csv")   # 최종 사용 환자 445명 (노트북 1 ⑥)
OUT_DIR = Path("../results/final_holter_ecg")
PLAN_PATH = OUT_DIR / "ecg_augmentation_plan.csv"                  # dry-run 결과 (파일 저장 없이 계산만)
META_PATH = OUT_DIR / "ecg_augmentation_metadata.csv"              # 만든 증강 파일 1개 = 1줄
ERROR_PATH = OUT_DIR / "ecg_augmentation_errors.csv"
VERIFY_PATH = OUT_DIR / "ecg_augmentation_verify.csv"
FIGURE_PATH = OUT_DIR / "ecg_augmentation_check.png"

# 증강 규칙 (요청서 값, 바꿀 수 있음)
MIN_CROP_HOURS = 15            # 증강 구간 길이 (시간). 중복을 가장 작게 하려고 이 길이 그대로 씀
NUM_AUGMENTATIONS = 2          # 원본당 최대 증강 개수 (1 또는 2)
MAX_OVERLAP_RATIO = None       # 두 증강 구간의 중복률(겹침 ÷ 합친 길이) 상한. None = 제한 없음
HIGH_OVERLAP_REPORT = 0.5      # 보고서에 '중복률 높음'으로 따로 적을 기준 (계산에는 영향 없음)

# 실행 스위치
RUN_FULL = True                # True일 때만 ⑦에서 445명 전체 파일을 만듦 (이미 있는 파일은 건너뜀). False면 계획·시험·검증만
N_TEST = 3                     # ⑥에서 먼저 만들어 볼 환자 수
CHECK_SECONDS = 60             # ⑧ 전체 검증에서 파일마다 비교할 구간 길이 (앞·가운데·끝 각각)

assert NUM_AUGMENTATIONS in (1, 2), "이 노트북은 증강 1개 또는 2개만 지원"
AUG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ② 원본 목록과 증강 대상: 원본만 고르고, 이미 만든 증강 파일은 대상에서 뺌
def list_original_ids():
    """PhysioNet RECORDS의 Holter 기록 중 .hea와 .dat가 둘 다 있는 것 = 원본"""
    names = [line.split("/")[-1] for line in RECORDS_FILE.read_text().split() if line.startswith("Holter_ECG/")]
    return [n for n in names if (RAW_DIR / f"{n}.hea").exists() and (RAW_DIR / f"{n}.dat").exists()]


def is_augmented_id(record_id, original_ids):
    """'원본ID-숫자' 형식이고 그 원본ID가 원본 목록에 있으면 증강 파일 (원본 ID에 하이픈이 있어도 구분됨)"""
    base, dash, suffix = record_id.rpartition("-")
    return bool(dash) and suffix.isdigit() and base in original_ids and record_id not in original_ids


original_ids = list_original_ids()
original_set = set(original_ids)
cohort = pd.read_csv(COHORT_PATH)

targets = [pid for pid in cohort["pid"] if pid in original_set and not is_augmented_id(pid, original_set)]
missing = sorted(set(cohort["pid"]) - original_set)
print(f"원본 {len(original_ids)}개 · 코호트 {len(cohort)}명 · 증강 대상 {len(targets)}명 · 원본 파일 없음 {len(missing)}명")

In [ ]:
# ③ 증강 구간 계획 (dry-run): 파일을 만들지 않고 헤더만 읽어 구간·중복·용량을 계산
SECONDS_PER_HOUR = 3600
BYTES_PER_SAMPLE = 2           # format 16 = 샘플 1개당 2바이트


def plan_crops(n_samples, fs):
    """원본 길이(샘플) → ([(시작, 끝), ...], 증강 못 하는 이유). 끝 샘플은 포함하지 않음"""
    crop_len = int(round(MIN_CROP_HOURS * SECONDS_PER_HOUR * fs))
    if n_samples < crop_len:
        return [], f"{MIN_CROP_HOURS}시간 미만"
    if n_samples == crop_len:
        return [], f"정확히 {MIN_CROP_HOURS}시간 (원본 복사가 됨)"
    if NUM_AUGMENTATIONS == 1:
        return [(0, crop_len)], ""
    crops = [(0, crop_len), (n_samples - crop_len, n_samples)]       # 맨 앞 15시간, 맨 뒤 15시간
    if MAX_OVERLAP_RATIO is not None and overlap_ratio(*crops) > MAX_OVERLAP_RATIO:
        return crops[:1], f"중복률이 {MAX_OVERLAP_RATIO} 초과 → 1개만"
    return crops, ""


def overlap_samples(crop_a, crop_b):
    """두 구간이 겹치는 샘플 수"""
    return max(0, min(crop_a[1], crop_b[1]) - max(crop_a[0], crop_b[0]))


def overlap_ratio(crop_a, crop_b):
    """중복률 = 겹친 길이 ÷ 두 구간을 합친 길이"""
    overlap = overlap_samples(crop_a, crop_b)
    union = (crop_a[1] - crop_a[0]) + (crop_b[1] - crop_b[0]) - overlap
    return overlap / union


def plan_record(pid):
    """환자 1명의 헤더 → 계획 표의 줄들 (증강 파일 1개 = 1줄, 증강 못 하면 이유 1줄)"""
    header = wfdb.rdheader(str(RAW_DIR / pid))
    fs, n = header.fs, header.sig_len
    crops, reason = plan_crops(n, fs)
    base = {"original_id": pid, "fs": fs, "n_sig": header.n_sig, "sig_name": "".join(header.sig_name),
            "original_samples": n, "original_duration_hours": n / fs / SECONDS_PER_HOUR, "reason": reason}
    if not crops:
        return [{**base, "augmented_id": None}]

    pair_overlap = overlap_samples(*crops) if len(crops) == 2 else 0
    theory = max(0, 2 * (crops[0][1] - crops[0][0]) - n) if len(crops) == 2 else 0   # L1 + L2 − T
    rows = []
    for k, (start, end) in enumerate(crops, start=1):
        rows.append({**base, "augmented_id": f"{pid}-{k}", "crop_start_sample": start, "crop_end_sample": end,
                     "overlap_samples": pair_overlap, "theoretical_min_overlap_samples": theory,
                     "overlap_ratio": overlap_ratio(*crops) if len(crops) == 2 else 0.0,
                     "estimated_bytes": (end - start) * header.n_sig * BYTES_PER_SAMPLE})
    return rows


plan = pd.DataFrame([row for pid in tqdm(targets, desc="헤더 읽기") for row in plan_record(pid)])
hours = lambda col: plan[col] / plan["fs"] / SECONDS_PER_HOUR
plan["crop_start_hours"] = hours("crop_start_sample")
plan["crop_end_hours"] = hours("crop_end_sample")
plan["crop_duration_hours"] = plan["crop_end_hours"] - plan["crop_start_hours"]
plan["overlap_hours"] = hours("overlap_samples")
plan["theoretical_min_overlap_hours"] = hours("theoretical_min_overlap_samples")
plan.to_csv(PLAN_PATH, index=False)

crops = plan.dropna(subset=["augmented_id"])
free_gb = shutil.disk_usage(AUG_DIR).free / 1e9
print(f"증강 파일 {len(crops)}개 예정 · 예상 용량 {crops['estimated_bytes'].sum() / 1e9:.1f} GB · 남은 공간 {free_gb:.0f} GB")
print("증강 못 하는 원본:", plan.loc[plan["augmented_id"].isna(), ["original_id", "original_duration_hours", "reason"]].to_dict("records"))
print("중복이 이론적 최소와 같은 비율:", (crops["overlap_samples"] == crops["theoretical_min_overlap_samples"]).mean())
display(crops.drop_duplicates("original_id")[["original_duration_hours", "overlap_hours", "overlap_ratio"]].describe().round(2))

In [ ]:
# ④ 증강 파일 1개 쓰기: 원본의 [시작, 끝) 구간을 정수 값 그대로 읽어 같은 형식으로 저장
def augmented_paths(aug_id):
    return AUG_DIR / f"{aug_id}.hea", AUG_DIR / f"{aug_id}.dat"


def write_crop(original_id, aug_id, start, end):
    """돌려주는 값: 'written' 또는 'exists' (이미 있으면 덮어쓰지 않음)"""
    if any(p.exists() for p in augmented_paths(aug_id)):
        return "exists"

    record = wfdb.rdrecord(str(RAW_DIR / original_id), sampfrom=start, sampto=end,
                           physical=False, return_res=16)            # 필요한 구간만, 16비트 정수 그대로
    TMP_DIR.mkdir(exist_ok=True)
    wfdb.wrsamp(aug_id, fs=record.fs, units=record.units, sig_name=record.sig_name,
                d_signal=record.d_signal, fmt=record.fmt, adc_gain=record.adc_gain,
                baseline=record.baseline, write_dir=str(TMP_DIR))
    for tmp_file in (TMP_DIR / f"{aug_id}.dat", TMP_DIR / f"{aug_id}.hea"):   # 다 쓴 뒤에 옮김 → 반쯤 쓴 파일이 남지 않음
        tmp_file.rename(AUG_DIR / tmp_file.name)
    return "written"


def file_fingerprint(path):
    """원본이 바뀌지 않았는지 볼 때 쓰는 (크기, 수정 시각)"""
    stat = path.stat()
    return stat.st_size, stat.st_mtime_ns


def sha256(path, chunk=1 << 24):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            digest.update(block)
    return digest.hexdigest()

In [ ]:
# ⑤ 검증 함수: 헤더 정보, 샘플 수, 파일 크기, 원본 같은 구간과의 정수 값 비교
def read_digital(path, start, end):
    return wfdb.rdrecord(str(path), sampfrom=start, sampto=end, physical=False, return_res=16).d_signal


def verify_crop(row, full_compare=False):
    """증강 파일 1개 검사 → 항목별 True/False. full_compare=False면 앞·가운데·끝 CHECK_SECONDS초씩만 비교"""
    hea, dat = augmented_paths(row.augmented_id)
    result = {"augmented_id": row.augmented_id, "files_exist": hea.exists() and dat.exists()}
    if not result["files_exist"]:
        return result

    org = wfdb.rdheader(str(RAW_DIR / row.original_id))
    aug = wfdb.rdheader(str(AUG_DIR / row.augmented_id))
    n = int(row.crop_end_sample - row.crop_start_sample)
    result.update({
        "fs_same": aug.fs == org.fs,
        "leads_same": aug.sig_name == org.sig_name,
        "gain_baseline_fmt_same": (aug.adc_gain == org.adc_gain) and (aug.baseline == org.baseline) and (aug.fmt == org.fmt),
        "header_samples_ok": aug.sig_len == n,
        "dat_size_ok": dat.stat().st_size == n * aug.n_sig * BYTES_PER_SAMPLE,
        "at_least_min_hours": n / aug.fs >= MIN_CROP_HOURS * SECONDS_PER_HOUR,
        "inside_original": 0 <= row.crop_start_sample < row.crop_end_sample <= org.sig_len,
    })

    if full_compare:
        pieces = [(0, n)]
    else:
        w = int(CHECK_SECONDS * aug.fs)
        pieces = [(0, w), (n // 2 - w // 2, n // 2 + w // 2), (n - w, n)]
    start = int(row.crop_start_sample)
    result["samples_same"] = all(
        np.array_equal(read_digital(AUG_DIR / row.augmented_id, a, b), read_digital(RAW_DIR / row.original_id, start + a, start + b))
        for a, b in pieces)
    return result

In [ ]:
# ⑥ 소수 시험: 24시간에 가까운 환자, 중간 길이 환자, 중복률이 가장 높은 환자 → 파일 만들기 → 전체 비교 → 그림
per_patient = crops.drop_duplicates("original_id").sort_values("original_duration_hours")
test_ids = list(dict.fromkeys([
    per_patient["original_id"].iloc[-1],                         # 가장 긴 기록
    per_patient["original_id"].iloc[len(per_patient) // 2],      # 중간 길이
    per_patient.sort_values("overlap_ratio")["original_id"].iloc[-1],   # 중복률 가장 높음
]))[:N_TEST]
test_rows = crops[crops["original_id"].isin(test_ids)]

before = {pid: (sha256(RAW_DIR / f"{pid}.dat"), sha256(RAW_DIR / f"{pid}.hea")) for pid in test_ids}
status = [write_crop(r.original_id, r.augmented_id, int(r.crop_start_sample), int(r.crop_end_sample)) for r in test_rows.itertuples()]
after = {pid: (sha256(RAW_DIR / f"{pid}.dat"), sha256(RAW_DIR / f"{pid}.hea")) for pid in test_ids}

test_check = pd.DataFrame([verify_crop(r, full_compare=True) for r in test_rows.itertuples()])
test_check.insert(1, "write", status)
display(test_rows[["original_id", "augmented_id", "original_duration_hours", "crop_start_hours", "crop_end_hours",
                   "overlap_hours", "theoretical_min_overlap_hours", "overlap_ratio"]].round(2))
display(test_check)
print("원본 sha256이 그대로인가:", before == after)
print("다시 쓰면 건너뛰는가:", all(write_crop(r.original_id, r.augmented_id, 0, 1) == "exists" for r in test_rows.itertuples()))

In [ ]:
# ⑥-2 그림: 증강 파일의 처음 5초를 원본의 같은 시각과 겹쳐 그림 (Y 유도, 물리값 mV). 선이 완전히 겹쳐야 정상
PLOT_SECONDS = 5
fig, axes = plt.subplots(len(test_rows), 1, figsize=(12, 2.2 * len(test_rows)), squeeze=False)
for ax, r in zip(axes[:, 0], test_rows.itertuples()):
    fs = int(r.fs)
    n = PLOT_SECONDS * fs
    start = int(r.crop_start_sample)
    original = wfdb.rdrecord(str(RAW_DIR / r.original_id), sampfrom=start, sampto=start + n, channel_names=["Y"]).p_signal[:, 0]
    augmented = wfdb.rdrecord(str(AUG_DIR / r.augmented_id), sampto=n, channel_names=["Y"]).p_signal[:, 0]
    t = np.arange(n) / fs
    ax.plot(t, original, lw=3, color="lightgray", label="원본 같은 시각")
    ax.plot(t, augmented, lw=0.8, color="tab:blue", label="증강 파일")
    ax.set_title(f"{r.augmented_id}: 원본 {r.crop_start_hours:.2f}h부터 · 최대 차이 {np.nanmax(np.abs(original - augmented)):.4f} mV", fontsize=9)
    ax.set_ylabel("mV")
axes[0, 0].legend(loc="upper right", fontsize=8)
axes[-1, 0].set_xlabel("초")
fig.tight_layout()
fig.savefig(FIGURE_PATH, dpi=120)
plt.show()

📝 **⑥ 시험 메모**
- 시험 환자와 구간:
- 검증 항목이 모두 True인가 / 원본 sha256 그대로인가 / 다시 쓰면 건너뛰는가:
- 그림에서 선이 겹치는가:

In [ ]:
# ⑦ 전체 실행 (RUN_FULL = True일 때만): 환자마다 증강 파일을 만들고, 오류가 나도 다음 환자로 넘어감
if RUN_FULL:
    fingerprints_before = {pid: (file_fingerprint(RAW_DIR / f"{pid}.dat"), file_fingerprint(RAW_DIR / f"{pid}.hea")) for pid in targets}
    statuses, errors = {}, []
    for r in tqdm(crops.itertuples(), total=len(crops), desc="증강 파일 쓰기", mininterval=30):
        try:
            statuses[r.augmented_id] = write_crop(r.original_id, r.augmented_id, int(r.crop_start_sample), int(r.crop_end_sample))
        except Exception as e:
            statuses[r.augmented_id] = "error"
            errors.append({"original_id": r.original_id, "augmented_id": r.augmented_id, "error": f"{type(e).__name__}: {e}"})
    pd.DataFrame(errors, columns=["original_id", "augmented_id", "error"]).to_csv(ERROR_PATH, index=False)
    fingerprints_after = {pid: (file_fingerprint(RAW_DIR / f"{pid}.dat"), file_fingerprint(RAW_DIR / f"{pid}.hea")) for pid in targets}
    print("쓰기 결과:", pd.Series(statuses).value_counts().to_dict(), "· 오류", len(errors))
    print("원본 파일(크기·수정 시각)이 그대로인가:", fingerprints_before == fingerprints_after)
else:
    print("RUN_FULL = False → 전체 실행 건너뜀 (⑥ 시험 결과를 확인한 뒤 True로 바꾸고 다시 실행)")

In [ ]:
# ⑧ 메타데이터 저장 + 전체 검증: 실제로 있는 증강 파일만 기록하고, 파일마다 헤더·크기·정수 값(앞·가운데·끝) 검사
existing = crops[[augmented_paths(a)[0].exists() for a in crops["augmented_id"]]].copy()
existing["num_samples"] = existing["crop_end_sample"] - existing["crop_start_sample"]
existing["sampling_frequency"] = existing["fs"]
existing["output_path"] = [str(AUG_DIR / a) for a in existing["augmented_id"]]
META_COLUMNS = ["original_id", "augmented_id", "original_duration_hours", "crop_start_hours", "crop_end_hours",
                "crop_duration_hours", "overlap_hours", "overlap_ratio", "theoretical_min_overlap_hours",
                "sampling_frequency", "num_samples", "output_path"]
existing[META_COLUMNS].to_csv(META_PATH, index=False)

verify = pd.DataFrame([verify_crop(r) for r in tqdm(existing.itertuples(), total=len(existing), desc="검증", mininterval=30)])
verify.to_csv(VERIFY_PATH, index=False)
checks = verify.drop(columns="augmented_id")
print(f"증강 파일 {len(existing)}개 · 메타데이터 저장 {META_PATH}")
display(checks.mean().to_frame("통과 비율").T)

leftover = sorted(set(p.stem for p in AUG_DIR.glob("*.hea")) - set(crops["augmented_id"]))
print("계획에 없는 증강 파일 (있으면 안 됨):", leftover)
print("증강 파일을 다시 증강 대상으로 잡았나:", any(is_augmented_id(t, original_set) for t in targets))

In [ ]:
# ⑨ 결과 요약: 요청서의 보고 항목
made = existing.drop_duplicates("original_id")
not_possible = plan[plan["augmented_id"].isna()]
size_gb = sum(p.stat().st_size for p in AUG_DIR.glob("*-*.*")) / 1e9
summary = {
    "원본 ECG (코호트)": len(targets),
    "증강 대상 ECG": crops["original_id"].nunique(),
    "증강 불가능 ECG": len(not_possible),
    "생성된 증강 ECG": len(existing),
    "원본 포함 총 ECG": len(targets) + len(existing),
    "증강 ECG 평균 길이 (h)": round(existing["crop_duration_hours"].mean(), 2) if len(existing) else None,
    "증강 파일 총용량 (GB)": round(size_gb, 1),
}
display(pd.Series(summary).to_frame("값"))
display(not_possible[["original_id", "original_duration_hours", "reason"]])
display(made[["overlap_hours", "overlap_ratio"]].agg(["mean", "min", "max"]).round(3))

high = made[made["overlap_ratio"] > HIGH_OVERLAP_REPORT].sort_values("overlap_ratio", ascending=False)
print(f"중복률 {HIGH_OVERLAP_REPORT} 초과 환자 {len(high)}명")
display(high[["original_id", "original_duration_hours", "overlap_hours", "overlap_ratio"]].round(3))

📝 **⑨ 결과 메모**
- 원본 / 증강 대상 / 증강 불가(이유) / 생성 개수 / 총 개수:
- 중복 시간과 중복률 (평균·최소·최대), 중복률 높은 환자:
- 총용량, 검증 통과 비율, 원본 그대로인지:
- 다음 단계: 증강 파일에 v2 전처리(특징 28개)를 돌릴 때 `RAW_DIR` 대신 `AUG_DIR`을 읽도록 해야 함